## ✅ ONNX/TensorRT Optimization

![Speedometer latency chart](https://cdn-icons-png.freepik.com/512/12710/12710013.png)

In this template, we optimized a PyTorch-based image classification model using **ONNX + TensorRT** for **fast, low-latency inference**, with minimal code changes. You trained a model using **AutoGluon’s MultiModalPredictor**, exported it to **ONNX format**, and ran **benchmark comparisons** between vanilla PyTorch inference and accelerated **TensorRT** inference — all inside a Jupyter Notebook.

Running this workflow on [**Saturn Cloud**](https://saturncloud.io) makes it both **GPU-accelerated and production-ready**. With Saturn Cloud, you can:

* 🔁 Train models using **multiple frameworks** like PyTorch, TensorFlow, Hugging Face, and AutoGluon.
* 🚀 Deploy on **scalable NVIDIA GPU machines** with support for CUDA, TensorRT, and ONNXRuntime.
* 📊 Run inference at scale using **interactive notebooks or scheduled jobs** — ideal for real-time applications.

This template gives you a full pipeline to **quantize, export, and benchmark** fast image inference — ideal for edge devices, cloud APIs, or any latency-sensitive AI application.

In [ ]:
# Install AutoGluon with multimodal and TensorRT support
!pip install -q autogluon.multimodal[all] onnx onnxruntime-gpu


### 📥 Step 1: Download and Prepare the PetFinder Dataset

This downloads a simplified **PetFinder** dataset used to train a model that predicts whether a pet is likely to be adopted quickly.

The dataset contains images + metadata (text, tabular, etc), but in this flow we’ll use **images only** for simplicity.

---

### 🧹 Step 2: Load and Clean the Data

Loads the CSV files and defines the key columns. We'll drop text/numerical data later to focus on image performance.

---



In [ ]:
import os
from autogluon.core.utils.loaders import load_zip

# 🔽 Download & unzip dataset
download_dir = './ag_automm_tutorial'
zip_url = 'https://automl-mm-bench.s3.amazonaws.com/petfinder_for_tutorial.zip'
load_zip.unzip(zip_url, unzip_dir=download_dir)

# ✅ Confirm structure
dataset_path = os.path.join(download_dir, 'petfinder_for_tutorial')
print("📁 Dataset folder:", dataset_path)
print("📷 Sample images:", os.listdir(os.path.join(dataset_path, 'images'))[:3])



### 🧹 Step 2: Load and Clean the Data

Loads the CSV files and defines the key columns. We'll drop text/numerical data later to focus on image performance.

---

### 🖼️ Step 3: Preprocess Image Paths

Ensures all image paths are **fully resolved** so AutoGluon can locate them.
---

In [ ]:
import pandas as pd
import os

# Define dataset path (same as from Step 1)
dataset_path = './ag_automm_tutorial/petfinder_for_tutorial'

# Load CSVs
train_data = pd.read_csv(f'{dataset_path}/train.csv', index_col=0)
test_data = pd.read_csv(f'{dataset_path}/test.csv', index_col=0)

# Target label
label_col = 'AdoptionSpeed'
image_col = 'Images'

# For this tutorial, we only use the first image in the list
train_data[image_col] = train_data[image_col].apply(lambda ele: ele.split(';')[0])
test_data[image_col] = test_data[image_col].apply(lambda ele: ele.split(';')[0])

# Expand image path
def path_expander(path, base_folder):
    return os.path.abspath(os.path.join(base_folder, path))

train_data[image_col] = train_data[image_col].apply(lambda x: path_expander(x, dataset_path))
test_data[image_col] = test_data[image_col].apply(lambda x: path_expander(x, dataset_path))

# 🔍 Preview
print("✅ Sample rows from training data:")
train_data[[image_col, label_col]].head()


### 🧠 Step 4: Train Image-Only Model with AutoGluon

Trains a lightweight image classification model (e.g., MobileNetV3) using only image data.

The model is trained quickly (~2 mins) to allow for demonstration and benchmarking.

---

In [ ]:
from autogluon.multimodal import MultiModalPredictor

# Drop extra columns – use image + label only
train_image_only = train_data[[image_col, label_col]]
test_image_only = test_data[[image_col, label_col]]

# Set a short training time (you can increase this later)
predictor = MultiModalPredictor(label=label_col).fit(
    train_data=train_image_only,
    time_limit=120,  # seconds
)

# Save model path
model_path = predictor.path
print(f"✅ Model trained and saved at: {model_path}")


### ⚙️ Step 5: Optimize the Model for Inference (TensorRT via ONNX)

Converts the model internally to ONNX format and enables **ONNX Runtime acceleration**.

---


In [ ]:
from autogluon.multimodal import MultiModalPredictor

# Load the model from previous training path
trt_predictor = MultiModalPredictor.load(path=model_path)

# Optimize for fast inference (uses ONNX + TensorRT if available)
trt_predictor.optimize_for_inference()

print("✅ Model optimized for ONNX/TensorRT inference!")


In [ ]:
import time
import numpy as np

# Use small batch for timing test
batch_size = 2
n_trials = 10
sample = test_image_only.head(batch_size)

# --- PyTorch inference timing ---
pt_times = []
for _ in range(n_trials):
    start = time.time()
    _ = predictor.predict_proba(sample)
    pt_times.append(time.time() - start)

pt_avg = np.mean(pt_times)
print(f"🐢 PyTorch Avg Time (per batch): {pt_avg*1000:.2f} ms")

# --- TensorRT (ONNX Runtime) inference timing ---
trt_times = []
for _ in range(n_trials):
    start = time.time()
    _ = trt_predictor.predict_proba(sample)
    trt_times.append(time.time() - start)

trt_avg = np.mean(trt_times)
print(f"⚡ TensorRT Avg Time (per batch): {trt_avg*1000:.2f} ms")


print("Final comparison of time lapse:")
print(f"🐢 PyTorch Avg Time (per batch): {pt_avg*1000:.2f} ms")
print(f"⚡ TensorRT Avg Time (per batch): {trt_avg*1000:.2f} ms")


### 🧪 Step 6: Compare Inference Speeds

Measures inference speed in **rows per second** across 10 runs.
Compares vanilla PyTorch inference with accelerated ONNX/TensorRT inference.

---

In [ ]:
import numpy as np

# Run predictions on the same sample
proba_pt = predictor.predict_proba(sample)
proba_trt = trt_predictor.predict_proba(sample)

# Show outputs
print("🎯 PyTorch Output:")
print(proba_pt)

print("\n🚀 TensorRT Output:")
print(proba_trt)

# Check if close (with small tolerance due to FP16 rounding)
try:
    np.testing.assert_allclose(proba_pt, proba_trt, rtol=1e-2, atol=1e-2)
    print("\n✅ Predictions are numerically close!")
except AssertionError as e:
    print("\n⚠️ Predictions differ more than expected.")
    print(str(e))


### 📊 Step 7: Visualize Speedup


Shows the real benefit of inference optimization — TensorRT is often **2–5x faster** than vanilla PyTorch.
Ideal for reducing latency in production.

---


In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np

# Define test batch
batch_size = 2
n_trials = 10
sample = test_data.head(batch_size)

# Measure PyTorch inference times
pt_times = []
for _ in range(n_trials):
    start = time.time()
    _ = predictor.predict_proba(sample)
    pt_times.append(time.time() - start)

# Measure TensorRT inference times
trt_times = []
for _ in range(n_trials):
    start = time.time()
    _ = trt_predictor.predict_proba(sample)
    trt_times.append(time.time() - start)

# Calculate rows per second
pt_speed = batch_size / np.mean(pt_times)
trt_speed = batch_size / np.mean(trt_times)

print(f"⚡ PyTorch speed: {pt_speed:.1f} rows/sec")
print(f"⚡ TensorRT speed: {trt_speed:.1f} rows/sec")

# Bar chart comparison
fig, ax = plt.subplots()
fig.set_figheight(1.5)
ax.barh(["PyTorch", "TensorRT"], [pt_speed, trt_speed])
ax.annotate(f"{pt_speed:.1f} rows/s", xy=(pt_speed, 0))
ax.annotate(f"{trt_speed:.1f} rows/s", xy=(trt_speed, 1))
_ = plt.xlabel("Inference Speed (rows per second)")
plt.show()




## ✅ Conclusion

In this template, we:

✅ Trained an image classifier using **AutoGluon’s MultiModalPredictor**
✅ Exported and optimized the model with **ONNX + TensorRT**
✅ Benchmarked inference performance side-by-side
✅ Verified minimal accuracy loss with **significant speed gains**

Running this end-to-end on **[Saturn Cloud](https://saturncloud.io)** ensures you have:

* 💻 Pre-configured NVIDIA GPU environments
* ⚡ Fast installation of CUDA, TensorRT, PyTorch, and ONNX
* 📅 Support for scheduled inference pipelines and model serving

Whether you're deploying to production or prototyping locally, **this template helps you productionize your models with speed and efficiency**.

---

### 📚 Continue Exploring

* [Saturn Cloud Documentation](https://saturncloud.io/docs/) – Custom environments, GPUs, and scheduling
* [ONNX + TensorRT Blog Post](https://saturncloud.io/blog/) – Coming soon
* [Saturn Cloud Templates](https://saturncloud.io/resources/templates/) – Other GPU-accelerated projects.
